# Disentanglement experiments on Colab

This notebook launches the repository's existing MSP or LibriSpeech trainer. It does not contain a second training implementation. A free runtime is one resumable segment, not a guarantee that a full experiment will finish in one session. Pilot results are development evidence only.

CLUB is used as a probe-architecture-agnostic objective. Its learned variational estimate is not, by itself, proof that every downstream probe fails; use the separate held-out probe phase.

In [ ]:
#@title 1. Experiment selection
DATASET = 'librispeech' #@param ['msp', 'librispeech']
EXPERIMENT = 'libri_grl_stats_gelu' #@param ['msp_baseline','msp_no_pcgrad','msp_no_invariance','msp_soft_routing','msp_no_cross_adversaries','msp_no_adversaries','libri_grl_stats_gelu','libri_club_hybrid','libri_club_pure']
PHASE = 'train' #@param ['train', 'probe']
PROFILE = 'full' #@param ['full', 'pilot']
SEED = 42 #@param {type:'integer'}
RUN_TAG = 'default' #@param {type:'string'}
SEGMENT_STEPS = 250 #@param {type:'integer'}
MAX_RUNTIME_MINUTES = 600 #@param {type:'integer'}
EFFECTIVE_BATCH_SIZE = 16 #@param {type:'integer'}
MICROBATCH_SIZE = 'auto' #@param {type:'string'}
RESUME_EVERY = 50 #@param {type:'integer'}
PRECISION = 'auto' #@param ['auto', 'bf16', 'fp16', 'fp32']
REPO_URL = 'https://github.com/beimnet777/Final-Proj.git' #@param {type:'string'}
GIT_COMMIT = 'main' #@param {type:'string'}
# Any hyperparameter registered by the selected trainer can be overridden here.
# Leave {} to use the named preset exactly. LibriSpeech uses 'stage2_steps'; MSP uses 'steps'.
OVERRIDES = {
    # --- schedule and optimizer ---
    # 'stage2_steps': 10000,
    # 'warmup_steps': 500,
    # 'lr': 1e-4,
    # 'lr_min': 1e-6,
    # 'lr_heads': 1e-4,
    # 'lr_sid_head': 5e-4,
    # 'lr_disc': 1e-3,
    # 'lr_routing': 1e-3,
    # 'weight_decay': 0.0,
    # 'eval_batch_size': 32,
    # 'grad_clip': 1.0,
    # 'n_disc_steps': 3,
    # --- primary and adversarial task weights ---
    # 'alpha': 0.8,
    # 'beta': 0.6,
    # 'grl_weight': 1.0,
    # 'grl_phoneme_weight': 0.15,
    # 'prosody_weight': 0.5,
    # 'grl_prosody_weight': 0.5,
    # 'emotion_weight': 0.5,
    # 'grl_emotion_weight': 0.5,
    # 'inv_weight': 1.0,
    # --- GRL architecture and normalization ---
    # 'grl_stats_pool': True,
    # 'grl_linear_stats': False,  # standalone signed mean+std GRL
    # 'grl_grad_norm': True,
    # 'grl_grad_norm_target': 2.5e-4,
    # 'grl_p_grad_norm': True,
    # 'grl_p_grad_norm_target': 1e-3,
    # 'dann_full_discriminator': True,
    # --- routing ---
    # 'gumbel_tau_start': 1.0,
    # 'gumbel_tau_end': 0.1,
    # 'routing_init_std': 0.5,
    # 'routing_spec_weight': 0.01,
    # 'rho': 0.0,
    # --- CLUB / probe-agnostic objective ---
    # 'club_enabled': True,
    # 'club_weight': 0.3,
    # 'club_lr': 1e-3,
    # 'club_inner_steps': 3,
    # 'club_hidden': 512,
    # 'club_grad_norm': True,
    # 'club_grad_norm_target': 0.005,
    # 'club_phoneme_enabled': True,
    # 'club_phoneme_weight': 0.3,
    # 'club_phoneme_lr': 1e-3,
    # 'club_phoneme_inner_steps': 3,
    # 'club_phoneme_hidden': 512,
    # 'club_phoneme_warmup_steps': 1000,
    # --- logging/checkpoints ---
    # 'ckpt_every': 1000,
    # 'log_every': 100,
    # 'grad_log_every': 500,
}
assert EXPERIMENT.startswith('msp_') == (DATASET == 'msp'), 'Dataset and experiment disagree'

In [ ]:
#@title 2. Mount Drive and clone the pinned source
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess
WORK = Path('/content/final-proj')
if not WORK.exists():
    token = ''
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN') or ''
    except Exception:
        pass
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    subprocess.run(['git', 'clone', url, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', GIT_COMMIT], check=True)
COMMIT = subprocess.check_output(['git','-C',str(WORK),'rev-parse','HEAD'], text=True).strip()
print('Pinned commit:', COMMIT)

In [ ]:
#@title 3. Install dependencies without replacing Colab PyTorch
import sys, subprocess, torch
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK/'Disentanglement/requirements-colab.txt')], check=True)
# g2p_en uses NLTK data files that pip cannot install.  New NLTK versions
# require the language-specific `*_eng` tagger used for LibriSpeech OOV words.
import nltk
NLTK_RESOURCES = {
    'averaged_perceptron_tagger': 'taggers/averaged_perceptron_tagger',
    'averaged_perceptron_tagger_eng': 'taggers/averaged_perceptron_tagger_eng',
    'cmudict': 'corpora/cmudict',
}
for package, resource in NLTK_RESOURCES.items():
    try:
        nltk.data.find(resource)
        print('NLTK resource ready:', package)
    except LookupError:
        print('Downloading NLTK resource:', package)
        nltk.download(package, quiet=True, raise_on_error=True)
        nltk.data.find(resource)
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, round(p.total_memory/2**30, 1), 'GiB', 'bf16', torch.cuda.is_bf16_supported())
subprocess.run(['df','-h','/content'])

In [ ]:
#@title 4. Download the complete project LibriSpeech dataset before testing
import shutil, subprocess, sys
DRIVE_ROOT = Path('/content/drive/MyDrive/FinalProjColab')
DATA_ROOT = Path('/content/data')
if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
if DATASET == 'librispeech':
    # The complete dataset used in this project: 100 h training plus clean dev/test.
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    drive_cache = DRIVE_ROOT/'assets'/'openslr'; drive_cache.mkdir(parents=True, exist_ok=True)
    for name in ('train-clean-100', 'dev-clean', 'test-clean'):
        cached_archive = drive_cache/f'{name}.tar.gz'
        local_archive = Path('/content')/f'{name}.tar.gz'
        if cached_archive.exists():
            print(f'Using cached Drive archive: {cached_archive.name}')
            shutil.copy2(cached_archive, local_archive)
        else:
            print(f'Downloading {name} from the official OpenSLR release ...')
            subprocess.run(['wget', '-c', '-O', str(local_archive), f'https://www.openslr.org/resources/12/{name}.tar.gz'], check=True)
            shutil.copy2(local_archive, cached_archive)
        subprocess.run(['tar', '-xzf', str(local_archive), '-C', str(DATA_ROOT)], check=True)
    shutil.copy2(WORK/'Probing/data/librispeech-lexicon.txt', DATA_ROOT/'librispeech-lexicon.txt')
    assert (DATA_ROOT/'LibriSpeech/train-clean-100').exists()
    assert (DATA_ROOT/'LibriSpeech/dev-clean').exists()
    assert (DATA_ROOT/'LibriSpeech/test-clean').exists()
else:
    asset_name = f'msp_{PROFILE}.tar.gz'
    source_archive = DRIVE_ROOT/'assets'/asset_name
    if not source_archive.exists():
        raise FileNotFoundError(f'MSP is licensed and cannot be downloaded automatically. Upload: {source_archive}')
    local_archive = Path('/content')/asset_name
    shutil.copy2(source_archive, local_archive)
    subprocess.run([sys.executable, '-m', 'Disentanglement.colab_bundle', 'verify', '--archive', str(local_archive), '--extract-to', str(DATA_ROOT)], cwd=WORK, check=True)
print('Data ready:', DATA_ROOT)

In [ ]:
#@title 5. Restore the Hugging Face cache locally (optional)
HF_HOME = Path('/content/hf')
cache_archive = DRIVE_ROOT/'assets'/'spear_cache.tar.gz'
if cache_archive.exists() and not HF_HOME.exists():
    subprocess.run(['tar','-xzf',str(cache_archive),'-C','/content'], check=True)
HF_HOME.mkdir(exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_CACHE'] = str(HF_HOME/'hub')

In [ ]:
#@title 6. Resolve and inspect the command (safe dry run)
import json
RUN_NAME = f'{EXPERIMENT}_{RUN_TAG}_s{SEED}'
LOCAL_RUN = Path('/content/runs')/RUN_NAME
DRIVE_RUN = DRIVE_ROOT/'runs'/EXPERIMENT/RUN_TAG/f'seed-{SEED}'
LOCAL_RUN.mkdir(parents=True, exist_ok=True); DRIVE_RUN.mkdir(parents=True, exist_ok=True)
base = [sys.executable, '-m', 'Disentanglement.experiment_runner', '--experiment', EXPERIMENT, '--data_root', str(DATA_ROOT), '--profile', PROFILE, '--phase', PHASE, '--seed', str(SEED), '--output_dir', str(LOCAL_RUN), '--drive_mirror', str(DRIVE_RUN), '--segment_steps', str(SEGMENT_STEPS), '--max_runtime_minutes', str(MAX_RUNTIME_MINUTES)]
if PHASE == 'train':
    base += ['--effective_batch_size',str(EFFECTIVE_BATCH_SIZE),'--microbatch_size',str(MICROBATCH_SIZE),'--resume','auto','--resume_every',str(RESUME_EVERY),'--precision',PRECISION]
    for key, value in OVERRIDES.items():
        base += ['--set', f'{key}={json.dumps(value)}']
subprocess.run(base + ['--dry_run'], cwd=WORK, check=True) if PHASE == 'train' else print('Probe command:', base)

In [ ]:
#@title 7. Restore checkpoint and run one segment with live logs
import datetime, os, shlex
for name in ('latest-resume.pt','best.pt','final.pt','metrics.jsonl'):
    src, dst = DRIVE_RUN/name, LOCAL_RUN/name
    if src.exists() and not dst.exists():
        print(f'Restoring {name} from Drive')
        shutil.copy2(src, dst)
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = LOCAL_RUN/f'segment_{stamp}.log'
print('Working directory:', WORK)
print('Run directory:', LOCAL_RUN)
print('Drive mirror:', DRIVE_RUN)
print('Command:', shlex.join(str(x) for x in base), flush=True)
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(base, cwd=WORK, env=env, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
tail = []
with log_path.open('w', encoding='utf-8') as log_file:
    for line in process.stdout:
        print(line, end='', flush=True)
        log_file.write(line); log_file.flush()
        tail.append(line.rstrip()); tail = tail[-80:]
return_code = process.wait()
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
shutil.copy2(log_path, DRIVE_RUN/log_path.name)
print(f'Full log saved to {log_path} and Drive (exit={return_code})')
if return_code != 0:
    print('\n===== LAST 80 LOG LINES =====')
    print('\n'.join(tail))
    raise RuntimeError(f'Training failed with exit code {return_code}; see {log_path}')

In [ ]:
#@title 8. Inspect persisted metrics
import json, pandas as pd, matplotlib.pyplot as plt
metrics = LOCAL_RUN/'metrics.jsonl'
if metrics.exists():
    rows = [json.loads(line) for line in metrics.read_text().splitlines() if line.strip()]
    frame = pd.DataFrame(rows); display(frame.tail())
    numeric = [c for c in frame.columns if c not in {'step','split'} and pd.api.types.is_numeric_dtype(frame[c])]
    if numeric: frame.plot(x='step', y=numeric[:8], subplots=True, figsize=(10, 2*min(8,len(numeric)))); plt.tight_layout()
else:
    print('No validation metrics yet; run another segment or reduce checkpoint interval.')